# Day 59 — Decision Trees and Impurity Metrics

This notebook covers Decision Tree Anatomy (Root, Internal Nodes, Leaves), Global vs. Local Partitioning, Gini Impurity, Information Gain, and Scikit-Learn Classification

## Validation goals
- Understand decision tree architecture and hierarchical binary splitting mechanics
- Differentiate global parametric models (linear and logistic regression) from local non-parametric axis-aligned partitioning
- Explore Gini impurity, entropy, and information gain as criteria for optimal node splitting
- IImplement a decision tree classifier in Python using Scikit-Learn and analyze overfitting risks in unconstrained trees

### 1. What is a Decision Tree?

### Concept Explanation

A Decision Tree is a supervised learning algorithm used for both classification and regression. It builds a model in the form of a tree structure, breaking down a dataset into smaller and smaller subsets while concurrently developing an associated decision tree incrementally.

### Anatomy of a Decision Tree

- Root Node: The starting point at the very top of the tree representing the entire dataset. It asks the single best question to split the data.
- Internal (Decision) Nodes: The branching points where further questions are asked based on specific feature values (e.g., "Is Age > 30?").
- Leaves (Terminal Nodes): The final endpoints of the tree that contain the ultimate prediction or class label (e.g., Approve Loan or Deny Loan).

**Relatable Example: Deciding Whether to Go for a Run**

Imagine deciding whether to go running on any given morning:
- Question 1 (Root): Is it raining?
    - Yes $\rightarrow$ Don't Run (Leaf).
    - No $\rightarrow$ Move to next question.
- Question 2: Is the temperature greater than 20°C?
    - Yes $\rightarrow$ Go Run (Leaf).
    - No $\rightarrow$ Wear a Jacket and Go Run (Leaf).

### 2. How the Algorithm Learns: Gini Impurity and Entropy

When building a tree, the computer doesn't manually guess which questions to ask. It tests every single feature and every possible threshold value to find the split that results in the purest possible groups.

To measure how "pure" a group is, algorithms use metrics like:
- Gini Impurity: Measures how often a randomly chosen element from a set would be incorrectly labeled if it were randomly labeled according to the distribution of labels in the set. A Gini score of 0.0 means a pure node (every item belongs to the exact same class).
- Information Gain (Entropy): Measures the reduction in uncertainty or randomness after a dataset is split.

The algorithm greedily chooses the feature split that maximizes information gain at every step.

### 3. Implementing a Decision Tree in Python (Scikit-Learn)

Let's write a clean Python script using Scikit-learn to train a DecisionTreeClassifier on a classic dataset (predicting whether a customer will purchase a product based on age and income).

In [2]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# 1. Setup synthetic dataset: [Age, Income ($k$)] vs. Purchased (0 = No, 1 = Yes)
X = np.array([
    [22, 25],
    [25, 30],
    [47, 85],
    [52, 90],
    [23, 20],
    [46, 110],
    [38, 60],
    [33, 45]
])
y = np.array([0, 0, 1, 1, 0, 1, 1, 0])  # Target Labels

# 2. Instantiate the Decision Tree Classifier
# max_depth limits how deep the tree can grow to prevent overfitting
model = DecisionTreeClassifier(max_depth=3, random_state=42)

# 3. Train (fit) the model on our data
model.fit(X, y)

print("--- Decision Tree Successfully Trained ---")
print(f"Number of features used: {model.n_features_in_}\n")

# 4. Make predictions for new, unseen customers
# Customer A: Age 28, Income $35k
# Customer B: Age 49, Income $95k
new_customers = np.array([
    [28, 35],
    [49, 95]
])

predictions = model.predict(new_customers)

for customer, pred in zip(new_customers, predictions):
    # Fixed ternary syntax
    decision = "Will Purchase" if pred == 1 else "Will Not Purchase"
    print(f"Customer (Age: {customer[0]}, Income: ${customer[1]}k) -> Prediction: {decision}")

--- Decision Tree Successfully Trained ---
Number of features used: 2

Customer (Age: 28, Income: $35k) -> Prediction: Will Not Purchase
Customer (Age: 49, Income: $95k) -> Prediction: Will Purchase


### Guided Question:

What is the fundamental difference in how a linear/logistic regression model makes predictions versus how a decision tree makes predictions?

### ANswer:

**Core Paradigm Shift: Global vs. Local Partitioning**

The fundamental difference lies in how each model views and carves up the feature space:

- Linear and Logistic Regression are global parametric models. They fit a single, continuous mathematical function across the entire feature space simultaneously, assuming a global relationship between inputs and outputs.
- Decision Trees are local non-parametric models (or hierarchical rule-based systems). They recursively partition the feature space into disjoint, axis-aligned regions (rectangles or hyperrectangles) using sequential conditional checks.

**1. Linear and Logistic Regression: The Global Hyperplane**

Linear and logistic regression models make predictions by taking a linear combination of input features, scaling each feature by a learned weight ($w$), and adding a bias intercept ($b$).
- Linear Regression Equation:$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = w^T x + b$$
- Logistic Regression (Classification): Passes the linear output through the sigmoid function $\sigma(z) = \frac{1}{1 + e^{-z}}$ to yield a probability score between 0 and 1.

**Key Characteristics:**

- Simultaneous Feature Contribution: Every feature contributes to every prediction at the exact same time through its respective coefficient.
- Global Decision Boundary: The boundary is a single rigid hyperplane (a straight line in 2D) defined across the entire dataset. Modifying a data point in one corner of the dataset subtly shifts the weights and alters predictions everywhere else.
- Extrapolation: Because it relies on an unbroken mathematical equation, it can extrapolate trends infinitely outside the range of the training data.

**2. Decision Trees: The Hierarchical Rule Engine**

Decision trees make predictions by routing an input vector down a flowchart of binary splitting rules learned during training (e.g., Is income > $50,000?).
- Prediction Mechanism: Starting at the root node, the feature value of the sample is evaluated against a threshold. Depending on whether the condition is true or false, the sample moves down the left or right branch until it hits a terminal leaf node.
- Leaf Output: The prediction is simply the average target value (for regression) or the majority class proportion (for classification) of the training samples that landed in that specific leaf.

**Key Characteristics:**

- Sequential Feature Selection: Not all features are used for every prediction. A path down the tree might test only 2 or 3 features out of 50.
- Axis-Aligned Partitioning: Decision boundaries are strictly orthogonal (parallel to the feature axes) because each split divides the space using a single feature at a time ($x_i \le c$).
- Local Constant Fitting: The model fits local constants to localized regions of the feature space rather than a global slope. It cannot extrapolate beyond the maximum or minimum target values seen in the training leaves.

### Guided Question:

What does a metric like Gini Impurity measure, and what does a Gini score of 0.0 represent for a node in a tree?

### Answer:

Gini impurity measures how often a randomly chosen element from a dataset or node would be incorrectly labeled if it were randomly labeled according to the distribution of class labels in that node. In practical terms, it quantifies the degree of mixedness, disorder, or heterogeneity of classes within a node.
- Formula:$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$where $C$ is the total number of classes and $p_i$ is the probability (the proportion) of items belonging to class $i$ in that node.
- Maximum Impurity: Occurs when classes are evenly distributed. For example, in a binary classification problem with a 50/50 split, the Gini impurity is $1 - (0.5^2 + 0.5^2) = 0.5$, representing maximum uncertainty.

**What Does a Gini Score of 0.0 Represent?**

A Gini score of `0.0` represents absolute purity (`homogeneity`). It means that every single data sample falling into that specific node belongs to the exact same class ($p_i = 1.0$ for one class and $0$ for all others). When a node reaches a Gini impurity of `0.0`, it is pure, meaning no further splits are required along that branch.Overfitting Risk: Why Unconstrained Decision Trees Overfit.

An unconstrained decision tree—one allowed to grow indefinitely without limits on depth or leaf size—has exceptionally high model capacity and non-parametric flexibility.

### Guided Question:

Why can an unconstrained decision tree (where max_depth is set to infinity) easily fall into the trap of overfitting your training data?

### Answer:

An unconstrained decision tree—where max_depth = None (infinity)—falls into the trap of overfitting because it possesses unbounded model capacity, allowing it to completely memorize the training data rather than learn generalizable underlying patterns.

Here is a breakdown of the specific mechanisms that cause an unconstrained tree to overfit:

**1. Recursive Splitting Until Complete Purity**

Without a depth limit, the tree will greedily continue splitting its feature space until every single leaf node reaches absolute homogeneity (e.g., a Gini impurity of 0.0). In practice, this means the tree will keep dividing branches until every individual data point (or tiny cluster of identical points) is isolated in its own leaf node.
- The Result: Training error drops to zero, but the model loses its ability to handle unseen data because it has engineered a rigid lookup table tailored exclusively to the training set.

**2. Capturing Noise as a True Signal**

Real-world datasets invariably contain noise—measurement errors, anomalous outliers, or mislabeled samples.
- An unconstrained tree treats every single data point as absolute truth. If a single outlier of class A sits right in the middle of a dense cluster of class B, the infinite-depth tree will carve out a tiny, hyper-specific rectangular pocket just to isolate that single noisy outlier.
- Instead of smoothing over random fluctuations, the model amplifies them into strict conditional rules.

**3. Extreme Model Variance**

Decision trees are notoriously high-variance models. Because the choice of the root node and early splits depends heavily on the exact composition of the dataset:
- Changing just a few rows in the training set of an unconstrained tree can cause the entire top structure of the tree to completely reorganize.
- This high sensitivity means the model fails to capture stable, robust trends that hold true across different samples of the same population.

**4. Fragmented, Jagged Decision Boundaries**

Because decision trees partition space using axis-aligned splits ($x_i \le c$), an unconstrained tree creates a piecewise constant, highly jagged step-function as a decision boundary.
- Instead of finding a clean, generalized linear or smooth curve, it carves the feature space into dozens of disjoint, irregular boxes that tightly hug every twist and turn of the training points. When exposed to test data just outside those tiny boxes, the model fails catastrophically.

**How Regularization Fixes This**

To prevent an unconstrained tree from overfitting, machine learning pipelines impose structural limits:
-` max_depth`: Caps the maximum number of splits from root to leaf, forcing the tree to learn broader, more generalized rules.
- `min_samples_split / min_samples_leaf`: Ensures a node must contain a meaningful number of samples before it is allowed to split or form a final leaf, preventing the isolation of statistical noise and outliers.